# Assessed Worksheet 4: Reinforcement Learning

### Instructions

This notebook is based on part of a larger effort to offer an approachable introduction to models of the mind and the brain, developed by [Jelle (aka Willem) Zuidema](https://staff.fnwi.uva.nl/w.zuidema/). The notebook in this present form is the result of the combined work of Iris Proff, [Marianne de Heer Kloots](http://mdhk.net/), and [Simone Astarita](https://www.linkedin.com/in/simone-astarita-4499b11b5/).


In this notebook, we look at how different parameters in Q-learning affect the behaviour of an agent.

### Introduction
In this notebook, we look at the behaviour of a Q-learning agent in a very simple environment, depicted below.

<div align='center'>
<img src="grid.jpg" width="200"/>
</div>

The agent starts in the leftmost state, which has reward 0.1. The rightmost state has reward 5, and is a terminal state. We will index these from left to right, so that the states are 0, 1, and 2 from left to right. There are four possible action directions, left (0), up (1), right (2) and down (3). At the edge of the gridworld the agent 'bounces back', i.e. the resulting state from taking action 'left' in state 0 is state 0.

You will use the code developed in the practice notebook to implement this gridworld and the Q-learning algorithm, and examine how the values of the parameters affect the behaviour of the agent.

### 1. Updating the Q-values

As the agent navigates through the maze, it builds up an estimation of the utility of each state-action pair. This estimation is represented in a 3x4 matrix $Q$. Each time the agent takes a step, the Q-value of the corresponding state-action pair is updated. Specifically, when moving from state $s$ to state $s'$ with action $a$ and obtaining reward $R_{s'}$, $Q(s,a)$ is updated according to:

$$
\begin{align*}
    Q_{t+1}(s,a)=Q_t(s,a)+\alpha*\delta
\end{align*}
$$

where $\delta$ is the prediction error, defined by:

$$
\begin{align*}
   \delta = R_{s'} + \gamma * max_{a'}(Q(s',a'))-Q(s,a)
\end{align*}
$$

Here, $\alpha$ is the learning rate and $\gamma$ is the temporal discount factor. $max_{a'}(Q(s',a'))$ refers to the highest Q-value of state $s'$. $Q(s,a)$ is updated proportionally to the size of the prediction error – the greater the prediction error, the more the agent learns.

In [ ]:
import numpy as np

def update_Q(a,s,s1,R,Q, gamma, alpha):
    """
    Function to update Q values.
    
    Input:
      a -- action (integer between 0 and 3)
      s -- state (integer between 0 and 15)
      s1 -- new state (integer between 0 and 15)
      R -- reward value
      Q -- (3, 4) array with Q-values for each (s, a) pair
      gamma -- temporal discount value
      alpha -- learning rate
      
    Output:
      Q[s, a] -- updated Q-value
      pred_error -- prediction error (delta)
    """
    
    # compute prediction error
    pred_error = R + gamma*np.nanmax(Q[s1,:])-Q[s, a]
    
    # update Q value
    Q[s,a] = Q[s,a]+alpha*pred_error
        
    return Q[s,a], pred_error


### 2. Softmax action selection

The second component of our Q-learning algorithm is an action selection function, that receives the Q-values of the current state as an input and returns an action to be taken. We will implement a softmax action selection function, that assigns probabilities to each action $a_i$ of a given state $s$, depending on its Q-value $q_i$:

$$
\begin{align*}
    P(q_i|s) = \frac{e^{\frac{q_i}{\tau}}}{\sum_A{e^{\frac{q_i}{\tau}}}}
\end{align*}
$$

Here, $\tau > 0$ is the so called temperature parameter. If $\tau$ is close to $0$, the algorithm most likely selects the action with the highest Q-value (i.e. it makes a *greedy* choice). For $\tau \rightarrow \infty$, the algorithm *randomly* selects one of the actions, irrespective of their Q-value. The softmax function is implemented in the cell below.

In [1]:
def softmax_act_select(Q, tau):
    """
    Softmax function for action selection.
    
    Input:
      Q -- (3, 4) array with Q-values for each (s, a) pair
      tau -- temperature parameter
    """
    
    Qs = Q[~np.isnan(Q)] # get valid actions
    actions =np.where(~np.isnan(Q)) # get valid action indices
    actions = actions[0]

    
    # compute probabliities for each action
    x = np.zeros(Qs.size); p = np.zeros(Qs.size)

    for i in range(Qs.size):
        x[i] = np.exp(Qs[i]/tau)/sum(np.exp(Qs/tau))

    p = x/sum(x)
    
    # choose action
    a = np.random.choice(actions, p = p)
    
    return a

### 3. Running the simulation

Now we are ready to run the simulation. The code below sets values for our model parameter and implements the maze structure. Then it runs the simulation. Our agent has to solve the maze 100 times (you can change this number). In each trial, it starts in the initial state and can move freely around in the maze until it reaches one of the terminal states. 

We store the number of steps the agent takes in each trial, the Q-values after each trial, the prediction errors and visited state of each step and which terminal state was reached in each trial. These results are plotted in the lower cell.

**Implementing the maze (2.5 pts)**
Fill in the code below to implement the gridworld given in the image above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### set parameter values

alpha = 0.1   # learning rate, 0 < alpha < 1
gamma = 0.5  # temporal discount factor, 0 <= gamma <=1
tau = 0.2    # temperature of softmax action selection, tau > 0
trials = 100  # number of times the agent has to solve the maze

### implement maze structure

# Initialize Q(s,a) with zeros for each state-action pair that is possible.
# Do include actions for terminal states (although these are not really possible)

# initialize Q(s,a)
# Your code here

# zeros for each possible action - include actions for terminal states
# Your code here

# Set up the initial and terminal states, and the rewards

# terminal and initial states
# Your code here

# rewards
# Your code here

### initialize variables to store data 
steps = np.zeros([trials,1])
s_term_meta = np.zeros([trials,1])
Q_meta = np.zeros([trials,3,4])
pred_error_meta = []
visited_states = []

states = np.arange(3).reshape(1,3)

### run simulation

for trial in range(trials):
    # print(f'Done {trial} trial(s)')
    
    # place agent in initial state
    s = s_init
    
    # store initial state
    visited_states.append([s_init])
    
    # store Q values
    Q_meta[trial,:,:] = Q
    
    # continue until in terminal state
    iters = 0
    while not(s in s_term) and iters < 200:
        iters +=1
        # print(f'State is: {s}')
        # choose action
        a = softmax_act_select(Q[s], tau)

        # observe new state
        # left
        if a == 0:
            if s == 0:
                s1 = s
            else:
                s1 = s-1
        # up
        elif a == 1:
            s1= s
        # right
        elif a == 2:
            if s == 2:
                s1 = s
            else:
                s1 = s+1
        # down
        else:
            s1 = s

        # observe R
        R = Rs[s1]

        # update Q
        Q[s,a], pred_error = update_Q(a,s,s1,R,Q, gamma, alpha)

        # update state
        s = s1
    
        # count steps
        steps[trial] += 1
        
        # store prediction error 
        pred_error_meta.append(pred_error)
        
        # store visited state
        visited_states[trial].append(s1)
    
    # store terminal state
    s_term_meta[trial] = s1
        

### plot some results

# plot final Q-values for each state
plt.figure(figsize=(8,4))
plt.imshow(Q)
cbar = plt.colorbar()
cbar.set_label('final Q-values')
plt.xticks([0,1,2,3], ['left', 'up', 'right', 'down'])
plt.yticks(range(3))
plt.xlabel('actions')
plt.ylabel('states')
plt.show()


###### helper funtions######

# function to get coordinates for given state (used for plotting)
def xy(s):
    x = np.where(states == s)[1][0]
    y = states.shape[0] - 1 - np.where(states == s)[0][0]
    return (x, y)

# function to plot visited states
def plot_map(visited_states):
    visited_path = np.array([xy(st) for st in visited_states])
    visited_unique = np.unique(visited_states)
    visited_xy = np.array([xy(st) for st in visited_unique])
    visited_counts = np.array([visited_states.count(st) for st in visited_unique])
    plt.scatter(visited_xy[:,0], visited_xy[:,1], s=visited_counts*100)
    plt.plot(visited_path[:,0], visited_path[:,1], 'k:', alpha=0.5)
    plt.xlim(-1,4); plt.ylim(-1,4); plt.xticks([]); plt.yticks([])
    for st in states.flatten():
        plt.annotate(st, xy(st))
        
##############################

# plot visited states
plot_trials = [0, 19, 59, 79, 99]
fig = plt.figure(figsize=(18,10))
for i in range(len(plot_trials)):
    ax = fig.add_subplot(1, len(plot_trials), i+1)
    plot_map(visited_states[plot_trials[i]])
    ax.set_title('trial ' + str(plot_trials[i]+1))
    ax.set_aspect(1)
plt.show()

# plot Qvalues over trials
fig, axes = plt.subplots()
s = 1; a = 2 # here you can choose which Q value to plot
plt.plot(Q_meta[:,s,a])
plt.xlabel('trials')
plt.ylabel('Q value({},{})'.format(s,a))
plt.show()

# plot prediction errors
fig, axes = plt.subplots()
plt.plot(pred_error_meta,'g')
plt.xlabel('steps')
plt.ylabel('prediction error')
plt.show()

# plot steps
fig, axes = plt.subplots()
plt.plot(steps,'r')
plt.xlabel('trials')
plt.ylabel('steps')
plt.show()

# plot terminal states
fig, axes = plt.subplots()
plt.plot(s_term_meta,'mx')
plt.xlabel('trials')
plt.ylabel('terminal state')
plt.yticks(s_term)
plt.show()

### 4. Homework exercises (7.5 pt)

> ***Homework exercise 1 (3pt)*** Describe and explain the evolution of (1) Q-values **(1pt)**, (2) prediction errors **(1pt)** and (3) step number over time **(1pt)** with the given parameter values.
>
> Hint: You can select which Q-value to plot in the code.


> ***Homework exercise 2 (1.5 pt)***
> Change the value of the reward in state 0 to 1. How does the behaviour change? **(0.5 pt)** Why does this happen? **(1 pt)**

> ***Homework exercise 3 (2 pt)***
> Now change the value of the reward in state 0 to 0.7. Assess what happens with different values of $\tau$, for example $\tau=0.01$ and $\tau=1$. What differences in behaviour do you see? **(1 pt)** Why do you see this? **(1 pt)** 

> ***Homework exercise 4 (1 pt)***
> Would you see the same difference in behaviour if the reward at state 0 were negative? Explain why, or why not. **(1 pt)** 
